# Fooocus 2.5.6 - Colab Edition

1. Comprueba que el entorno tenga GPU: **Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU**.
2. Ejecuta la celda de abajo.
3. Espera el enlace `Running on public URL: https://xxxx.gradio.live` y abrelo.

La primera vez descarga unos 9 GB de modelos (checkpoint + inpaint + expansion).
Activa `CACHEAR_MODELOS_EN_DRIVE` si quieres que esa descarga pase una sola vez.

In [ ]:
# @title Arrancar Fooocus
BRANCH = 'colab-2.5.6'  # @param {type:"string"}
CACHEAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
EXTRAS_OPCIONALES = False  # @param {type:"boolean"}
ARGUMENTOS_EXTRA = '--always-high-vram'  # @param {type:"string"}

import os
import shlex
import shutil
import subprocess
import sys

REPO = 'https://github.com/deleonramiro085/Fooocus.git'
WORKDIR = '/content/Fooocus'
DRIVE_CACHE = '/content/drive/MyDrive/Fooocus/models'
SHARED_SUBDIRS = ['checkpoints', 'loras', 'inpaint', 'controlnet', 'clip_vision',
                  'upscale_models', 'vae', 'vae_approx', 'sam', 'safety_checker']

gpu = ''
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    pass
print('GPU:', gpu or 'NO DETECTADA')
if not gpu:
    print('Sin GPU no hay nada que hacer: Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU.')

if not os.path.isdir(os.path.join(WORKDIR, '.git')):
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, WORKDIR], check=True)
os.chdir(WORKDIR)

if CACHEAR_MODELOS_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in SHARED_SUBDIRS:
        target = os.path.join(DRIVE_CACHE, sub)
        os.makedirs(target, exist_ok=True)
        local = os.path.join(WORKDIR, 'models', sub)
        if os.path.islink(local):
            continue
        shutil.rmtree(local, ignore_errors=True)
        os.symlink(target, local)
    print('Modelos cacheados en', DRIVE_CACHE)

cmd = [sys.executable, '-u', 'entry_with_update.py', '--skip-update', '--share',
       '--preset', 'default', '--disable-analytics']
if EXTRAS_OPCIONALES:
    cmd.append('--install-optional')
cmd += shlex.split(ARGUMENTOS_EXTRA)
print(' '.join(cmd))
subprocess.run(cmd, check=False)


In [ ]:
# @title Diagnostico (solo si algo falla)
import importlib.metadata as md
import platform

print('python', platform.python_version())
try:
    import torch
    print('torch', torch.__version__, '| cuda', torch.version.cuda)
    print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
except Exception as e:
    print('torch no importable:', e)
for p in ('gradio', 'gradio_client', 'numpy', 'transformers', 'tokenizers', 'huggingface_hub',
          'pydantic', 'fastapi', 'starlette', 'websockets', 'python-multipart',
          'onnxruntime', 'pygit2', 'pillow'):
    try:
        print(p, md.version(p))
    except Exception:
        print(p, 'no instalado')


## Si algo va mal

- **No sale el enlace `gradio.live`**: paciencia, el primer arranque son 3-8 minutos.
- **`CUDA out of memory`**: borra `--always-high-vram` de `ARGUMENTOS_EXTRA`.
- **Empezar de cero**: borra la carpeta `/content/Fooocus` y vuelve a ejecutar.
- **Mascara automatica / quitar fondo**: activa `EXTRAS_OPCIONALES` (rembg, SAM, GroundingDINO).
- **Ver que versiones hay instaladas**: ejecuta la celda de diagnostico.